# NeuTraL-AD Adaptive Transformations — Colab Experiment Runner

This notebook runs both the **Baseline** and **Adaptive** NeuTraL-AD experiments on the NATOPS dataset.

**Before running:**
- Go to **Runtime → Change runtime type → T4 GPU**
- Then run all cells top-to-bottom

**Results are saved to:**
- `RESULTS/RESULTS_BASELINE_natops/` — original fixed-K NeuTraL-AD
- `RESULTS/RESULTS_ADAPTIVE_natops/` — adaptive transformation weighting
- Google Drive (optional, Cell 7)

In [ ]:
# ─── CELL 1: Check GPU ──────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('WARNING: No GPU found! Training will be very slow on CPU.')
    print('Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ─── CELL 2: Clone repository ───────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/Subash1607/SSLADWNT.git'
REPO_DIR  = 'SSLADWNT'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest changes...')
    os.chdir(REPO_DIR)
    os.system('git pull origin main')
else:
    os.system(f'git clone {REPO_URL}')
    os.chdir(REPO_DIR)

print('\nCurrent directory:', os.getcwd())
print('Git log (last 5 commits):')
os.system('git log --oneline -5')

In [ ]:
# ─── CELL 3: Install dependencies ───────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'scikit-learn', 'scipy', 'pyyaml', '--quiet'], check=True)
print('Dependencies installed.')

In [ ]:
# ─── CELL 4: Verify data files ──────────────────────────────────────────────
import os
import numpy as np

files = [
    'DATA/natops/train_array.npy',
    'DATA/natops/train_label.npy',
    'DATA/natops/test_array.npy',
    'DATA/natops/test_label.npy',
]
for f in files:
    if os.path.exists(f):
        arr = np.load(f)
        print(f'  ✓ {f:<40} shape={arr.shape}  dtype={arr.dtype}')
    else:
        print(f'  ✗ MISSING: {f}')

In [ ]:
# ─── CELL 5: Run Sanity Check ───────────────────────────────────────────────
# Verifies the adaptive implementation is correct before running full experiments.
# Takes < 2 minutes.
print('Running sanity check...')
result = os.system('python colab_sanity_check.py')
if result != 0:
    print('\n[!] SANITY CHECK FAILED — do not proceed with full experiments!')
    print('[!] Report the error output above.')
else:
    print('\n[✓] Sanity check passed — safe to run full experiments.')

In [ ]:
# ─── CELL 6A: Run BASELINE Experiment ──────────────────────────────────────
# Original NeuTraL-AD, 6 classes, 100 epochs, K=11, adaptive=False
# Expected time: ~20-30 minutes on T4 GPU
print('='*60)
print('  BASELINE: Fixed-K NeuTraL-AD (adaptive=False)')
print('='*60)
os.makedirs('RESULTS', exist_ok=True)

# Delete previous baseline results if re-running (comment out to resume)
# os.system('rm -rf RESULTS/RESULTS_BASELINE_natops')

os.system('python Launch_Exps.py --config-file config_files/config_natops_gpu.yml --dataset-name natops')

In [ ]:
# ─── CELL 6B: Run ADAPTIVE Experiment ──────────────────────────────────────
# Adaptive NeuTraL-AD, 6 classes, 100 epochs, K=11, adaptive=True
# Expected time: ~25-35 minutes on T4 GPU (slightly slower due to scorer)
print('='*60)
print('  ADAPTIVE: Transformation Importance Scoring (adaptive=True)')
print('='*60)

# Delete previous adaptive results if re-running (comment out to resume)
# os.system('rm -rf RESULTS/RESULTS_ADAPTIVE_natops')

os.system('python Launch_Exps.py --config-file config_files/config_natops_adaptive_gpu.yml --dataset-name natops')

In [ ]:
# ─── CELL 6C: Compare Results ───────────────────────────────────────────────
print('\nComparing baseline vs adaptive results...')
os.system('python compare_results.py --baseline RESULTS/RESULTS_BASELINE_ --adaptive RESULTS/RESULTS_ADAPTIVE_ --dataset natops')

In [ ]:
# ─── CELL 7: Save Results to Google Drive (IMPORTANT — prevents loss!) ──────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/NeuTraLAD_Results'
os.makedirs(DRIVE_PATH, exist_ok=True)

import shutil, datetime
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dest = f'{DRIVE_PATH}/run_{timestamp}'
shutil.copytree('RESULTS', dest)
print(f'Results saved to Drive: {dest}')

# Also list what was saved
for root, dirs, files_l in os.walk(dest):
    level = root.replace(dest, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files_l:
        print(f'{indent}  {f}')

In [ ]:
# ─── CELL 8: Quick Result Viewer ────────────────────────────────────────────
import json
import os

def print_assessment(folder, label):
    path = os.path.join(folder, 'assessment_results.json')
    if not os.path.exists(path):
        print(f'  [{label}] No assessment_results.json found at {path}')
        return
    with open(path) as f:
        d = json.load(f)
    print(f'\n{label} Results:')
    print(f"  Mean AUROC: {d.get('avg_TS_auc_all', 'N/A'):.4f}")
    print(f"  Mean AP:    {d.get('avg_TS_ap_all',  'N/A'):.4f}")
    print(f"  Mean F1:    {d.get('avg_TS_f1_all',  'N/A'):.4f}")
    print()
    for i in range(6):
        auc = d.get(f'avg_TS_auc_{i}', None)
        ap  = d.get(f'avg_TS_ap_{i}',  None)
        f1  = d.get(f'avg_TS_f1_{i}',  None)
        if auc is not None:
            print(f'  Class {i+1}: AUROC={auc:.4f}  AP={ap:.4f}  F1={f1:.4f}')

print_assessment('RESULTS/RESULTS_BASELINE_natops',  'BASELINE')
print_assessment('RESULTS/RESULTS_ADAPTIVE_natops',  'ADAPTIVE')